In [2]:
import pandas as pd
import networkx as nx
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import graph
import ast

Import Bills data

In [3]:
billsPath =  os.path.join('..', 'data', 'raw', 'final_bill_data.csv')
billsDF = pd.read_csv(billsPath, engine='python')
billsDF = billsDF[billsDF['municipality'] == 'Boston']
billsDF

,bill_number,bill_url,title,text,sponsor,sponsor_url,view_text_url,download_pdf_url,is_home_rule,municipality,num_cosponsors,cosponsors,history,path
125,H.5097,https://malegislature.gov/Bills/192/H5097,An Act relative to certain affordable housing ...,SECTION 1. Construction and development activi...,"Michlewitz, Aaron",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/192/H5097/Hous...,https://malegislature.gov/Bills/192/H5097.pdf,True,Boston,0,[],"[{'date': '7/26/2022', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...
294,H.5039,https://malegislature.gov/Bills/192/H5039,An Act relative to Chapter 30B: Uniform Procur...,"In the City of Boston, notwithstanding section...","Holmes, Russell E.",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/192/H5039/Hous...,https://malegislature.gov/Bills/192/H5039.pdf,True,Boston,0,[],"[{'date': '7/18/2022', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...
567,H.3753,https://malegislature.gov/Bills/192/H3753,An Act regarding preference for Boston high sc...,"SECTION 1. In this act, all terms shall have t...","Fluker Oakley, Brandy",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/192/H3753/Hous...,https://malegislature.gov/Bills/192/H3753.pdf,True,Boston,0,[],"[{'date': '5/10/2021', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...
1058,H.5330,https://malegislature.gov/Bills/192/H5330,An Act regarding the disability pension for El...,Section 1. (a) Notwithstanding any general or ...,"Hunt, Daniel J.",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/192/H5330/Hous...,https://malegislature.gov/Bills/192/H5330.pdf,True,Boston,0,[],"[{'date': '9/29/2022', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...
2347,H.4111,https://malegislature.gov/Bills/192/H4111,An Act authorizing the city of Boston to estab...,Section 1. Paragraph (i) of subdivision (4) of...,"Miranda, Liz",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/192/H4111/Hous...,https://malegislature.gov/Bills/192/H4111.pdf,True,Boston,16,"[{'name': 'Thomas M. Stanley', 'district': '9t...","[{'date': '9/16/2021', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63907,H.3540,https://malegislature.gov/Bills/190/H3540,An Act relative to certain affordable housing ...,SECTION 1. Construction and development activi...,"Rushing, Byron",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/190/H3540/Hous...,https://malegislature.gov/Bills/190/H3540.pdf,True,Boston,0,[],"[{'date': '4/13/2017', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/190th/2025/...
64008,H.3541,https://malegislature.gov/Bills/190/H3541,An Act to restore Boston's governmentally-invo...,"SECTION 1. Whereas, a serious public emergency...","Rushing, Byron",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/190/H3541/Hous...,https://malegislature.gov/Bills/190/H3541.pdf,True,Boston,4,"[{'name': 'Jeffrey Sánchez', 'district': '15th...","[{'date': '4/13/2017', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/190th/2025/...
64633,HD.4192,https://malegislature.gov/Bills/190/HD4192,An Act to designate the courthouse in the Roxb...,The courthouse located at 85 Warren Street in ...,"Tyler, Chynah",https://malegislature.gov/Legislators/Profile/...,https://malegislature.gov/Bills/190/HD4192/Hou...,https://malegislature.gov/Bills/190/HD4192.pdf,True,Boston,0,[],"[{'date': '8/17/2017', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/190th/2025/...
66092,H.4324,https://malegislature.gov/Bills/190/H4324,An Act authorizing the Economic Development an...,SECTION 1. (a) The Economic Development and In...,"Collins, Nick",https://male

Import legislator data

In [5]:
legislatorsPath =  os.path.join('..', 'data', 'raw', 'extended_legislators.csv')
legislatorsDF = pd.read_csv(legislatorsPath, engine='python')
legislatorsDF

,url,First Name,Last Name,District,Party,committee_ids
0,https://malegislature.gov/Legislators/Profile/...,James,Arciero,2nd Middlesex,Democrat,[30]
1,https://malegislature.gov/Legislators/Profile/...,James,Arena-DeRosa,8th Middlesex,Democrat,"[1, 16, 23, 31]"
2,https://malegislature.gov/Legislators/Profile/...,Jennifer Balinsky,Armini,8th Essex,Democrat,"[12, 27, 30, 49]"
3,https://malegislature.gov/Legislators/Profile/...,Shirley,Arriaga,8th Hampden,Democrat,"[15, 21, 31, 32, 55]"
4,https://malegislature.gov/Legislators/Profile/...,Brian,Ashe,2nd Hampden,Democrat,"[5, 29, 45, 54]"
...,...,...,...,...,...,...
448,https://malegislature.gov/Legislators/Profile/...,Marie,Fleur,NaN,NaN,NaN
449,https://malegislature.gov/Legislators/Profile/...,Richard,Ross,NaN,NaN,NaN
450,https://malegislature.gov/Legislators/Profile/...,Robert,Nyman,NaN,NaN,NaN
451,https://malegislature.gov/Legislators/Profile/...,William,Lantigua,NaN,NaN,NaN


In [14]:
# Mapping each bill and legislator to a unique node id
legislator_ids = list(legislatorsDF['url'].unique())
legislator_id_map = {url: idx for idx, url in enumerate(legislator_ids)}

bill_ids = list(billsDF['bill_url'])
bill_id_map = {url: idx + len(legislator_ids) for idx, url in enumerate(bill_ids)}

In [7]:
print(legislator_id_map)
print(bill_id_map)

{'https://malegislature.gov/Legislators/Profile/J_A1': 0, 'https://malegislature.gov/Legislators/Profile/JCD1': 1, 'https://malegislature.gov/Legislators/Profile/JBA1': 2, 'https://malegislature.gov/Legislators/Profile/SBA1': 3, 'https://malegislature.gov/Legislators/Profile/BMA1': 4, 'https://malegislature.gov/Legislators/Profile/BJA1': 5, 'https://malegislature.gov/Legislators/Profile/MLB1': 6, 'https://malegislature.gov/Legislators/Profile/CPB2': 7, 'https://malegislature.gov/Legislators/Profile/J_B1': 8, 'https://malegislature.gov/Legislators/Profile/DRB1': 9, 'https://malegislature.gov/Legislators/Profile/D_B1': 10, 'https://malegislature.gov/Legislators/Profile/NMB1': 11, 'https://malegislature.gov/Legislators/Profile/NAG1': 12, 'https://malegislature.gov/Legislators/Profile/AFC1': 13, 'https://malegislature.gov/Legislators/Profile/DFC1': 14, 'https://malegislature.gov/Legislators/Profile/S_C1': 15, 'https://malegislature.gov/Legislators/Profile/MSC1': 16, 'https://malegislature.

In [8]:

# Custom graph implemntation, adding bill nodes

mygraph = graph.CustomGraph()
for bill_url, bill_id  in bill_id_map.items():
    bill = billsDF[billsDF['bill_url'] == bill_url]
    billNode = graph.BillNode(bill_id, bill['bill_number'], bill_url, bill['title'], bill['text'], bill['sponsor'], bill['sponsor_url'], bill['view_text_url'], bill['download_pdf_url'], bill['is_home_rule'], bill['municipality'], bill['num_cosponsors'], bill['cosponsors'], bill['history'], bill['path'])
    mygraph.add_bill(billNode)


In [9]:
# Custom graph implementation, adding legislator nodes
for legislator_url, legislator_id  in legislator_id_map.items():
    legislator = legislatorsDF[legislatorsDF['url'] == legislator_url]
    legislatorNode = graph.LegislatorNode(legislator_id, legislator_url, legislator['First Name'], legislator['Last Name'], legislator['District'], legislator['Party'], legislator['committee_ids'] )
    mygraph.add_legislator(legislatorNode)

In [10]:
mygraph.edges = []
for bill_id, bill in mygraph.bill_nodes.items():
    sponsor = bill.sponsor
    sponsor_url = bill.sponsor_url.iloc[0]
    #print(sponsor_url)
    sponsor_id = None

    #adding bill <-> sponsor
    if sponsor_url in legislator_id_map.keys():
        sponsor_id = legislator_id_map[sponsor_url]
        mygraph.add_edge(bill_id, sponsor_id, 'sponsor')
    #adding bill <-> cosponsor
    cosponsors_raw = bill.cosponsors.iloc[0]
    cosponsors = []
    if isinstance(cosponsors_raw, str):
        cosponsors = ast.literal_eval(cosponsors_raw)
    else:
        cosponsors = cosponsors_raw  # already parsed
    if len(cosponsors) > 0:
        for cosponsor in cosponsors:
            # print(cosponsor)
            cosponsor_url = cosponsor['url'].split('/1')[0]
            #print(cosponsor_url)
            if cosponsor_url in legislator_id_map.keys():
                cosponsor_id = legislator_id_map[cosponsor_url]
                mygraph.add_edge(bill_id, cosponsor_id, 'cosponsor')
    

In [11]:
print("Total edges:", len(mygraph.edges))
print("Sample edges:")
for edge in mygraph.edges[:5]:
    print(edge)


Total edges: 412
Sample edges:
(453, 95, 'sponsor')
(454, 61, 'sponsor')
(455, 41, 'sponsor')
(456, 66, 'sponsor')
(457, 341, 'sponsor')


In [12]:
def get_subgraph(mygraph, root_id, depth=2):
    from collections import deque

    visited = set()
    sub_nodes = set()
    sub_edges = []

    queue = deque([(root_id, 0)])

    while queue:
        node_id, d = queue.popleft()
        if node_id not in visited:
            
            if d > depth:
                break
            visited.add(node_id)
            sub_nodes.add(node_id)

            for src, dst, rel in mygraph.edges:
                #print(src, dst, rel)
                if src == node_id or dst == node_id:
                    neighbor = dst if src == node_id else src
                    if neighbor not in visited and d + 1 <= depth:
                        sub_edges.append((src, dst, rel))
                        #print("Appending", neighbor)
                        queue.append((neighbor, d + 1))
        #else:
            #print("Node already visited")
    return sub_nodes, sub_edges


In [27]:
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

def visualize_with_networkx(mygraph, root_id, depth=1):
    G = nx.DiGraph()

    sub_nodes, sub_edges = get_subgraph(mygraph, root_id, depth)
    #print(sub_nodes, sub_edges)

    G.add_node(str(root_id), label="Bill " + str(root_id), type='bill')

    # Add nodes
    print("Adding nodes")
    for node_id in sub_nodes:
        if node_id in mygraph.bill_nodes:
            bill_name = str(mygraph.bill_nodes[node_id].title.to_string())
            G.add_node(str(node_id), label="Bill " +  str(node_id), title = bill_name ,type='bill')
        elif node_id in mygraph.legislator_nodes:
            #legislator_id_map[node_id]
            first = str(mygraph.legislator_nodes[node_id].first.to_string().split()[1])
            last = str(mygraph.legislator_nodes[node_id].last.to_string().split()[1])
            party = str(mygraph.legislator_nodes[node_id].party.to_string().split()[1])
            nameAndParty = first + " " + last + ", " + party
            G.add_node(str(node_id), label="Legislator " + str(node_id), title =  (nameAndParty) , type='legislator')
    
    # Add edges
    print("Adding edges")
    for src, dst, rel in sub_edges:
        G.add_edge(str(src), str(dst), label=rel)


    # filter only for the nodes with the required degree
    min_degree = 0
    nodes_to_keep = [node for node, degree in G.degree() if degree >= min_degree]
    pruned_subgraph = G.subgraph(nodes_to_keep)

    # Draw
    print("Drawing")
    net = Network(height="800px", width="100%", notebook=True, bgcolor="#222222", font_color="white", cdn_resources="remote"  )
    net.force_atlas_2based(gravity=-200, central_gravity=0.01, spring_length=100)
    #net.toggle_physics(False)
    net.from_nx(pruned_subgraph)

    net.repulsion(node_distance=400, central_gravity=0.1)

    # Save and view
    # need to install jinja2 and pyvis
    net.show("graph.html")


sample_bill_id = bill_id_map['https://malegislature.gov/Bills/192/H5097']
print(sample_bill_id)
visualize_with_networkx(mygraph, sample_bill_id, 10)


453
Adding nodes
Adding edges
Drawing
graph.html
